<a href="https://colab.research.google.com/github/kuds/mesozoic-labs/blob/main/notebooks/jax_training.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# JAX/MJX Dinosaur Training (Colab)

Train a dinosaur species using **MuJoCo MJX** (JAX-accelerated physics) with a
from-scratch PPO implementation in pure JAX. MJX supports configurable batched
simulation and is designed to improve throughput on a compatible GPU. The project
does not yet publish a controlled cross-backend benchmark.

**Runtime:** A CUDA-capable Colab GPU is intended for full runs. Use CPU only for small smoke tests.

**Supported Species:**
- `trex` — T-Rex: balance → locomotion → bite
- `velociraptor` — Raptor: balance → locomotion → strike
- `brachiosaurus` — Brachio: balance → locomotion → food reach

Set `SPECIES` in the configuration cell below to choose.

In [ ]:
# Install dependencies and verify GPU
!pip install mujoco==3.10.0 mujoco-mjx==3.10.0 "jax[cuda12]" flax optax

import os
import subprocess

GPU_AVAILABLE = subprocess.run("nvidia-smi").returncode == 0
if not GPU_AVAILABLE:
    print("GPU not found; continue only with a small CPU smoke-test configuration.")

NVIDIA_ICD_CONFIG_PATH = "/usr/share/glvnd/egl_vendor.d/10_nvidia.json"
if not os.path.exists(NVIDIA_ICD_CONFIG_PATH):
    with open(NVIDIA_ICD_CONFIG_PATH, "w") as f:
        f.write('{"file_format_version":"1.0.0","ICD":{"library_path":"libEGL_nvidia.so.0"}}')

os.environ["MUJOCO_GL"] = "egl" if GPU_AVAILABLE else os.environ.get("MUJOCO_GL", "osmesa")

import jax
import mujoco

print(f"JAX devices: {jax.devices()}")
print(f"MuJoCo: {mujoco.__version__}")
print("Setup complete.")

In [ ]:
# GPU diagnostics — run this anytime to check utilization
# (especially useful to run from a second terminal during training)
import subprocess

result = subprocess.run(
    [
        "nvidia-smi",
        "--query-gpu=name,utilization.gpu,utilization.memory,memory.used,memory.total,temperature.gpu,power.draw",
        "--format=csv,noheader,nounits",
    ],
    capture_output=True,
    text=True,
    env={**__import__("os").environ, "COLUMNS": "200"},
)
if result.returncode == 0:
    lines = result.stdout.strip().split("\n")
    for line in lines:
        parts = [p.strip() for p in line.split(",")]
        if len(parts) >= 7:
            print(f"GPU:         {parts[0]}")
            print(f"Utilization: {parts[1]} % (compute)  {parts[2]} % (memory)")
            print(f"Memory:      {parts[3]} MiB / {parts[4]} MiB")
            print(f"Temperature: {parts[5]}   Power: {parts[6]} W")
        else:
            print(line)
else:
    print("nvidia-smi failed:", result.stderr)

# Tip: You can also run this from a Colab terminal:
#   watch -n 2 nvidia-smi

In [ ]:
# Clone mesozoic-labs and install with JAX extras
# (On Colab/Kaggle, clones the repo; locally, assumes the notebook lives inside the repo)
import os
import sys

_COLAB = "COLAB_GPU" in os.environ or os.path.exists("/content")
_KAGGLE = "KAGGLE_KERNEL_RUN_TYPE" in os.environ or os.path.exists("/kaggle")
if _COLAB:
    !git clone https://github.com/kuds/mesozoic-labs.git /content/mesozoic-labs 2>/dev/null || echo 'Already cloned'
    !pip install -e "/content/mesozoic-labs[jax]" -q
    _REPO_ROOT = "/content/mesozoic-labs"
elif _KAGGLE:
    !git clone https://github.com/kuds/mesozoic-labs.git /kaggle/working/mesozoic-labs 2>/dev/null || echo 'Already cloned'
    !pip install -e "/kaggle/working/mesozoic-labs[jax]" -q
    _REPO_ROOT = "/kaggle/working/mesozoic-labs"
else:
    # Running locally — walk up from cwd until we find pyproject.toml
    _REPO_ROOT = os.getcwd()
    _d = _REPO_ROOT
    while _d != os.path.dirname(_d):
        if os.path.isfile(os.path.join(_d, "pyproject.toml")):
            _REPO_ROOT = _d
            break
        _d = os.path.dirname(_d)

# Ensure the repo root is on sys.path so `from environments.…` works
# even if the editable install did not fully resolve.
if _REPO_ROOT not in sys.path:
    sys.path.insert(0, _REPO_ROOT)

# Verify the import path is correct
if not os.path.isdir(os.path.join(_REPO_ROOT, "environments", "shared")):
    raise RuntimeError(
        f"Could not find 'environments/shared' under {_REPO_ROOT}. "
        "If running outside the repo, clone it first:\n"
        "  !git clone https://github.com/kuds/mesozoic-labs.git && "
        "pip install -e mesozoic-labs[jax]"
    )

from IPython.display import clear_output

clear_output()
print(f"mesozoic-labs[jax] ready (repo: {_REPO_ROOT})")

In [ ]:
import logging
import time

# Silence JAX compilation spam (tracing/compiling/XLA warnings every update)
logging.getLogger("jax._src.dispatch").setLevel(logging.ERROR)
logging.getLogger("jax._src.interpreters.pxla").setLevel(logging.ERROR)

import jax
import jax.numpy as jnp
import mujoco

from environments.shared.jax_checkpoint import load_checkpoint
from environments.shared.jax_normalization import (
    RunningMeanStd,
    decay_running_stats,
    normalize_obs,
)
from environments.shared.jax_ppo import (
    PPOConfig,
    make_actor_critic,
    make_optimizer,
)

# Shared modules from mesozoic-labs
from environments.shared.jax_viz import (
    create_frame_collage,
    plot_locomotion_diagnostics,
    plot_reward_components,
    plot_training_curves,
    record_training_video,
)
from environments.shared.plant_contract import current_plant_identity

print(f"JAX {jax.__version__}, devices: {jax.devices()}")
print(f"MuJoCo {mujoco.__version__}")

if jax.devices()[0].platform == "gpu":
    print("GPU acceleration available")
else:
    print("CPU backend active; keep the environment batch and update count small.")

In [ ]:
# ============================================================
# USER CONFIGURATION
# ============================================================
SPECIES = "trex"  # Choose: "trex", "velociraptor", "brachiosaurus", "dibothrosuchus"
CURRENT_STAGE = 1  # Curriculum stage: 1=balance, 2=locomotion, 3=species-specific
USE_GOOGLE_DRIVE = True  # Set to True to save outputs to Google Drive (persistent across sessions)
VERBOSE = 1  # 0=eval/summary only, 1=periodic updates (default), 2=every update

# Explicit seed roles. Keep these fixed across stages for a reproducible run.
NETWORK_SEED = 42
TRAINING_SEED = 0
EVALUATION_SEED = 42

# In a fresh Colab session, paste the prior Drive directory timestamp here
# to continue stages 2 or 3. Blank reuses this runtime's active run.
RUN_ID = ""
if not RUN_ID:
    RUN_ID = globals().get("_ACTIVE_RUN_ID", "")

# Stage 1 starts fresh. Stages 2/3 auto-resume the prior stage's gated best model.
# An explicit relative path is resolved from RUN_DIR (for example,
# "stage1/models/best_model.pkl"); use an absolute path only when necessary.
RESUME_FROM = None
ALLOW_LEGACY_PLANT = False  # explicit migration-only override

# ============================================================
# Setup via shared library (replaces ~80 lines of hardcoded config)
# ============================================================
from environments.shared.jax_setup import (
    create_env,
    make_obs_fn,
    make_reward_fns,
    make_scale_action_fn,
    print_species_summary,
    setup_output_dirs,
    setup_species,
)

ctx = setup_species(SPECIES, stage=CURRENT_STAGE)
PLANT_IDENTITY = current_plant_identity(SPECIES)

# Storage directories
if USE_GOOGLE_DRIVE:
    from google.colab import drive

    drive.mount("/content/drive")
    _STORAGE_ROOT = "/content/drive/MyDrive/mesozoic-labs/logs"
else:
    _STORAGE_ROOT = "logs"

_dirs = setup_output_dirs(
    SPECIES,
    CURRENT_STAGE,
    storage_root=_STORAGE_ROOT,
    timestamp=RUN_ID or None,
)
RUN_DIR = _dirs["run_dir"]
STAGE_DIR = _dirs["stage_dir"]
MODEL_DIR = _dirs["model_dir"]
OUTPUT_DIR = STAGE_DIR  # backward compat
_ACTIVE_RUN_ID = RUN_DIR.name
RUN_ID = _ACTIVE_RUN_ID

# Capture immutable provenance before any training begins. Repeated calls
# against the same RUN_ID are idempotent and reject incompatible run settings.
from environments.shared.result_bundle import initialize_result_bundle

PROVENANCE_PATH = initialize_result_bundle(
    RUN_DIR,
    species=SPECIES,
    algorithm="JAX_PPO",
    backend="jax-mjx",
    seed=TRAINING_SEED,
    evaluation_seeds=[EVALUATION_SEED],
    seed_roles={
        "network": NETWORK_SEED,
        "training": TRAINING_SEED,
        "publication_evaluation": EVALUATION_SEED,
    },
    evaluation_episodes=30,
    parallel_envs=int(ctx.jax_kwargs.get("num_envs", 2048)),
    hardware=f"Google Colab ({jax.devices()[0]})" if _COLAB else str(jax.devices()[0]),
    plant_identity=PLANT_IDENTITY.to_dict(),
    run_id=RUN_ID,
    repository_root=_REPO_ROOT,
)

print_species_summary(ctx)
print(f"Run ID:     {RUN_ID}")
print(f"Run dir:    {RUN_DIR}")
print(f"Stage dir:  {STAGE_DIR}")
print(f"Provenance: {PROVENANCE_PATH}")
print(f"Drive:      {USE_GOOGLE_DRIVE}")
if RESUME_FROM:
    print(f"Resuming from checkpoint: {RESUME_FROM}")

## 1. Load Model into MJX

In [ ]:
# Model already loaded by setup_species() — just display info
mj_model = ctx.mj_model
print(f"{SPECIES.title()} model loaded:")
print(f"  Bodies: {mj_model.nbody}, Joints: {mj_model.njnt}, Actuators: {mj_model.nu}")
print(f"  qpos: {mj_model.nq}, qvel: {mj_model.nv}, Geoms: {mj_model.ngeom}")
print(f"  Total mass: {sum(mj_model.body_mass):.2f} kg, Timestep: {mj_model.opt.timestep * 1000:.1f} ms")

## 2. Environment Functions
Pure-JAX functions for observation, reward, reset, and step. These use shared
task/config concepts and selected primitives from `environments.shared`, while
the Gymnasium (SB3) and MJX (JAX) paths retain separate wiring and backend-specific
Stage-3 success semantics.

**Note:** Some NumPy-facing helpers use `float()` / Python `if`, which break
`jax.vmap` tracing. The JAX-compatible reward and termination implementation lives
in the shared backend library and is obtained below through `make_reward_fns(ctx)`.

In [ ]:
# ---------- Constants (resolved by setup_species) ----------
# All body/site IDs, sensor layout, termination checks, and TOML config
# are now in ctx (SpeciesContext). Extract frequently-used values for
# backward compatibility with downstream cells.

_stage_cfg = ctx.stage_config
_jax_kw = ctx.jax_kwargs
reward_cfg = ctx.reward_cfg
SENSOR_LAYOUT = ctx.sensor_layout
ROOT_BODY_ID = ctx.root_body_id
OBS_DIM = ctx.obs_dim
ACT_DIM = ctx.act_dim
CTRL_RANGE = ctx.ctrl_range
NUM_ENVS = _jax_kw.get("num_envs", 2048)
FRAME_SKIP = ctx.frame_skip
MAX_EPISODE_STEPS = ctx.max_episode_steps
HEALTHY_Z_MIN, HEALTHY_Z_MAX = ctx.healthy_z_range
MAX_TILT_ANGLE = ctx.max_tilt_angle
NATURAL_FORWARD_Z = ctx.natural_forward_z
_NOSEDIVE_TERMINATION_THRESHOLD = reward_cfg.get("nosedive_termination_threshold", 0.5)

print(f"Obs dim: {OBS_DIM}, Act dim: {ACT_DIM}")
print(f"Termination: {len(ctx.termination_body_checks)} body, {len(ctx.termination_site_checks)} site checks")
print(f"Reward keys: {[k for k, v in reward_cfg.items() if isinstance(v, (int, float)) and v != 0]}")

In [ ]:
# Observation and action functions (from library)
get_obs = make_obs_fn(ctx)
scale_action = make_scale_action_fn(ctx)

print(f"Observation dim: {OBS_DIM}")
print(f"Action dim: {ACT_DIM}")

## 3. Batched MJX Step

A single `jax.jit`-compiled function that steps `N` parallel environments.

In [ ]:
# Create the canonical MJXDinoEnv via the library helper (merges env_kwargs)
env = create_env(ctx, num_envs=NUM_ENVS)
mjx_model = env.mjx_model

# Bind rewards only after create_env resolves the stage-specific MJX config.
compute_reward, compute_reward_detailed, is_terminated = make_reward_fns(ctx)

print(f"MJXDinoEnv created: {SPECIES} stage {CURRENT_STAGE}")
print(f"  num_envs={NUM_ENVS}, action_dim={env.action_dim}, frame_skip={env.config.frame_skip}")
print(f"  fall_penalty={env.config.fall_penalty}, max_steps={env.config.max_episode_steps}")
print(f"  reward keys: {list(env.config.reward_weights.keys())}")

## 4. Policy Network (Flax)
Uses the shared `ActorCritic` from `environments.shared.jax_ppo`.

In [ ]:
from pathlib import Path

# Initialize using the shared ActorCritic from jax_ppo
network = make_actor_critic(action_dim=ACT_DIM)
rng = jax.random.PRNGKey(NETWORK_SEED)
dummy_obs = jnp.zeros((OBS_DIM,))
params = network.init(rng, dummy_obs)

# Observation normalization (stabilizes training across species/stages)
obs_rms = RunningMeanStd.create(OBS_DIM)

# Resolve continuation before loading any checkpoint. Stages 2/3 default to
# the previous stage's selected model, but only after its persisted gate passes.
_resume_update = 0
_jax_kw_resume = {}
try:
    from environments.shared.config import load_stage_config as _lsc

    _jax_kw_resume = _lsc(SPECIES, CURRENT_STAGE).get("jax_kwargs", {})
except Exception:
    pass

_ckpt_path = None
_auto_resume = CURRENT_STAGE > 1 and RESUME_FROM is None
if _auto_resume:
    import json

    _previous_stage = CURRENT_STAGE - 1
    _previous_stage_dir = RUN_DIR / f"stage{_previous_stage}"
    _previous_result_path = _previous_stage_dir / "stage_result.json"
    if not _previous_result_path.is_file():
        raise FileNotFoundError(
            f"Cannot auto-resume Stage {CURRENT_STAGE}: missing {_previous_result_path}. "
            f"Run and save Stage {_previous_stage} under RUN_ID={RUN_ID!r} first."
        )
    _previous_result = json.loads(_previous_result_path.read_text())
    if _previous_result.get("publication_gate_passed") is not True:
        raise RuntimeError(
            f"Cannot auto-resume Stage {CURRENT_STAGE}: Stage {_previous_stage} "
            "does not have publication_gate_passed=true in stage_result.json."
        )
    _ckpt_path = _previous_stage_dir / "models" / "best_model.pkl"
    if not _ckpt_path.is_file():
        raise FileNotFoundError(f"Previous-stage best model is missing: {_ckpt_path}")
    print(f"Auto-resuming from gated Stage {_previous_stage}: {_ckpt_path}")
elif RESUME_FROM is not None:
    _ckpt_path = Path(RESUME_FROM).expanduser()
    if not _ckpt_path.is_absolute():
        _ckpt_path = RUN_DIR / _ckpt_path
    if not _ckpt_path.is_file():
        raise FileNotFoundError(f"Requested checkpoint is missing: {_ckpt_path}")

if _ckpt_path is not None:
    _ckpt = load_checkpoint(
        _ckpt_path,
        current_plant=PLANT_IDENTITY,
        allow_legacy_plant=ALLOW_LEGACY_PLANT,
    )
    params = jax.device_put(_ckpt["params"])
    _resume_update = _ckpt.get("update", 0)
    if "obs_rms" in _ckpt:
        obs_rms = _ckpt["obs_rms"]
        # When transitioning between stages, the obs distribution shifts
        # (e.g. near-zero velocity in balance → sustained velocity in
        # locomotion).  With 2048 envs the prior count can be ~65M, making
        # update_running_stats nearly a no-op.  Decay the count so new
        # data adapts the statistics within a few updates.
        _obs_decay = _jax_kw_resume.get("obs_rms_decay_on_resume", 0.01)
        if _obs_decay < 1.0:
            _old_count = obs_rms.count
            obs_rms = decay_running_stats(obs_rms, decay_factor=_obs_decay)
            print(f"  obs_rms count decayed: {_old_count:,.0f} → {obs_rms.count:,.0f} (factor={_obs_decay})")
    print(f"Resumed from {_ckpt_path} (update {_resume_update})")
    if "reward_history" in _ckpt:
        print(
            f"  Prior history: {len(_ckpt['reward_history'])} updates, best reward: {max(_ckpt['reward_history']):.4f}"
        )

n_params = sum(p.size for p in jax.tree.leaves(params))
print(f"ActorCritic parameters: {n_params:,}")

## 5. PPO Implementation
Core PPO functions (`sample_action`, `compute_gae`, `ppo_loss`) are now
imported from `environments.shared.jax_ppo`. Notebook-specific wrappers
below adapt them for the training loop.

In [ ]:
# PPO wrapper functions now built internally by jax_trainer module.
# (sample_action, ppo_loss, scan_ppo_epochs are JIT-compiled in train())
print("PPO functions handled by jax_trainer module.")

## 6. Training Loop

In [ ]:
# ---------- Hyperparameters (loaded from TOML [jax] section) ----------

# JAX/MJX training hyperparameters (from TOML, with notebook defaults as fallback)
NUM_ENVS = _jax_kw.get("num_envs", 2048)
ROLLOUT_LEN = _jax_kw.get("rollout_len", 64)
NUM_UPDATES = _jax_kw.get("num_updates", 500)
PPO_EPOCHS = _jax_kw.get("ppo_epochs", 4)
MINIBATCH_SIZE = _jax_kw.get("minibatch_size", 512)
LEARNING_RATE = _jax_kw.get("learning_rate", 3e-4)
MAX_GRAD_NORM = _jax_kw.get("max_grad_norm", 0.5)
GAMMA = _jax_kw.get("gamma", 0.99)
GAE_LAMBDA = _jax_kw.get("gae_lambda", 0.95)
CLIP_RANGE = _jax_kw.get("clip_range", 0.2)
ENT_COEF = _jax_kw.get("ent_coef", 0.01)
FALL_PENALTY = _jax_kw.get("fall_penalty", -10.0)
RESET_NOISE_SCALE = _jax_kw.get("reset_noise_scale", 0.05)
INIT_QPOS_NOISE = _jax_kw.get("init_qpos_noise", 0.01)
INIT_YAW_NOISE = _jax_kw.get("init_yaw_noise", 0.1)

# Curriculum warmup (constrains policy updates while critic adapts to new reward landscape)
WARMUP_UPDATES = _jax_kw.get("warmup_updates", 0)  # 0 = no warmup (Stage 1 default)
WARMUP_CLIP_RANGE = _jax_kw.get("warmup_clip_range", 0.02)
WARMUP_ENT_COEF = _jax_kw.get("warmup_ent_coef", 0.02)

# Reward ramp (linearly ramp a reward weight from a fraction to its full value)
RAMP_UPDATES = _jax_kw.get("ramp_updates", 0)  # 0 = no ramp (Stage 1 default)
RAMP_ATTR = _jax_kw.get("ramp_attr", "forward_vel_weight")
RAMP_START_FRACTION = _jax_kw.get("ramp_start_fraction", 0.1)

print("Training config (from TOML [jax] section):")
print(f"  Species: {SPECIES}")
print(f"  Envs: {NUM_ENVS}")
print(f"  Rollout length: {ROLLOUT_LEN}")
print(f"  Updates: {NUM_UPDATES}")
print(f"  Total env steps: {NUM_ENVS * ROLLOUT_LEN * NUM_UPDATES:,}")
print(f"  Stage: {CURRENT_STAGE} ({ctx.stage_name})")
print(f"  Learning rate: {LEARNING_RATE}")
print(f"  Gamma: {GAMMA}")
print(f"  Clip range: {CLIP_RANGE}")
print(f"  Entropy coef: {ENT_COEF}")
print(f"  Max grad norm: {MAX_GRAD_NORM}")
print(f"  Fall penalty: {FALL_PENALTY}")
print(f"  Reset noise: joints={RESET_NOISE_SCALE}, xy={INIT_QPOS_NOISE}, yaw={INIT_YAW_NOISE}")
if WARMUP_UPDATES > 0:
    print(f"  Warmup: {WARMUP_UPDATES} updates (clip={WARMUP_CLIP_RANGE}, ent={WARMUP_ENT_COEF})")
if RAMP_UPDATES > 0:
    print(f"  Reward ramp: {RAMP_ATTR} from {RAMP_START_FRACTION:.0%} to 100% over {RAMP_UPDATES} updates")
print(f"  Reward config: {reward_cfg}")

In [ ]:
# Reward and termination wrappers were bound after MJXDinoEnv creation above.
# compute_reward, compute_reward_detailed, is_terminated are ready to use.
print(f"Active reward components: {[k for k, v in reward_cfg.items() if isinstance(v, (int, float)) and v != 0]}")

In [ ]:
# Initialize environments using MJXDinoEnv.reset()
# Use a different seed from network initialization to avoid correlated streams.
rng = jax.random.PRNGKey(TRAINING_SEED)
rng, reset_rng = jax.random.split(rng)
states = env.reset(reset_rng)

print(f"Initialized {NUM_ENVS} parallel environments via MJXDinoEnv.reset().")
print(f"Obs shape: {states.obs.shape}")
print(
    f"Reset noise: joints=±{env.config.reset_noise_scale}, "
    f"xy=±{env.config.init_qpos_noise}m, yaw=±{env.config.init_yaw_noise}rad"
)

In [ ]:
# Optimizer (with gradient clipping and LR decay)

LEARNING_RATE_END = _jax_kw.get("learning_rate_end", None)
VF_COEF = _jax_kw.get("vf_coef", 0.5)
VF_CLIP_RANGE = _jax_kw.get("vf_clip_range", None)
TARGET_KL = _jax_kw.get("target_kl", 0.05)

ppo_config = PPOConfig(
    learning_rate=LEARNING_RATE,
    learning_rate_end=LEARNING_RATE_END,
    max_grad_norm=MAX_GRAD_NORM,
    total_updates=NUM_UPDATES,
    n_epochs=PPO_EPOCHS,
    clip_range=CLIP_RANGE,
    vf_clip_range=VF_CLIP_RANGE,
    vf_coef=VF_COEF,
    ent_coef=ENT_COEF,
    gamma=GAMMA,
    gae_lambda=GAE_LAMBDA,
    target_kl=TARGET_KL,
)
optimizer = make_optimizer(ppo_config)
opt_state = optimizer.init(params)

if LEARNING_RATE_END is not None:
    print(f"LR schedule: {LEARNING_RATE} -> {LEARNING_RATE_END} (linear decay over {NUM_UPDATES * PPO_EPOCHS} steps)")
if VF_CLIP_RANGE is not None:
    print(f"Value function clipping: +/- {VF_CLIP_RANGE}")
if TARGET_KL is not None:
    print(f"KL early stopping: target_kl={TARGET_KL}")

print(f"Value function coefficient: {VF_COEF}")
print("Optimizer created (PPO updates handled by jax_trainer module).")

In [ ]:
# ---------- Main Training Loop (via shared jax_trainer module) ----------
from environments.shared.jax_trainer import TrainConfig, train

train_config = TrainConfig(
    # Core PPO
    num_envs=NUM_ENVS,
    rollout_len=ROLLOUT_LEN,
    num_updates=NUM_UPDATES,
    ppo_epochs=PPO_EPOCHS,
    minibatch_size=MINIBATCH_SIZE,
    learning_rate=LEARNING_RATE,
    learning_rate_end=LEARNING_RATE_END,
    max_grad_norm=MAX_GRAD_NORM,
    gamma=GAMMA,
    gae_lambda=GAE_LAMBDA,
    clip_range=CLIP_RANGE,
    vf_coef=VF_COEF,
    ent_coef=ENT_COEF,
    vf_clip_range=VF_CLIP_RANGE,
    target_kl=TARGET_KL,
    # Dimensions
    obs_dim=OBS_DIM,
    act_dim=ACT_DIM,
    # Curriculum
    warmup_updates=WARMUP_UPDATES,
    warmup_clip_range=WARMUP_CLIP_RANGE,
    warmup_ent_coef=WARMUP_ENT_COEF,
    ramp_updates=RAMP_UPDATES,
    ramp_attr=RAMP_ATTR,
    ramp_start_fraction=RAMP_START_FRACTION,
    # Checkpointing & logging
    checkpoint_freq=25,
    max_checkpoints=5,
    verbose=VERBOSE,
    # Output paths
    output_dir=STAGE_DIR,
    model_dir=MODEL_DIR,
    # Resume
    start_update=_resume_update,
    # Metadata
    species=SPECIES,
    stage=CURRENT_STAGE,
    frame_skip=FRAME_SKIP,
)


# Reward component diagnostics function (optional)
def _reward_detail_fn(data, action):
    return compute_reward_detailed(data, action, reward_cfg)


rng = jax.random.PRNGKey(TRAINING_SEED)

result = train(
    config=train_config,
    env=env,
    network=network,
    params=params,
    opt_state=opt_state,
    obs_rms=obs_rms,
    reward_cfg=reward_cfg,
    rng=rng,
    reward_detail_fn=_reward_detail_fn,
    optimizer=optimizer,
)

# Unpack results for downstream cells (plotting, eval, artifacts)
params = result.params
obs_rms = result.obs_rms
best_params = result.best_params
best_reward = result.best_reward
best_update = result.best_update
reward_history = result.reward_history
loss_history = result.loss_history
episode_return_history = result.episode_return_history
diagnostics_history = result.diagnostics_history
reward_component_history = result.reward_component_history
total_steps = result.total_steps
elapsed = result.elapsed
csv_path = result.csv_path

## Stage Gate Evaluation

Run full evaluation episodes on CPU to check whether the curriculum gate
thresholds (reward, episode length, forward velocity, and task success, where
configured) have been met. This notebook checks them once after training; it
does not implement SB3's consecutive early-advancement checks.

In [ ]:
# ---------- Stage Gate Evaluation (via shared library) ----------
from environments.shared.jax_setup import print_eval_summary, run_stage_evaluation

eval_results, final_eval_results, stage_results, gate_passed, gate_failures = run_stage_evaluation(
    ctx,
    env,
    params,
    network,
    obs_rms,
    n_episodes=30,
    best_params=best_params,
    best_reward=best_reward,
    best_update=best_update,
    total_steps=total_steps if "total_steps" in dir() else 0,
    elapsed=elapsed if "elapsed" in dir() else 0.0,
    eval_seed=EVALUATION_SEED,
)

# Expose diagnostic variables for downstream plotting cells
diag_tilt = eval_results.diag_tilt
diag_fwd_vel = eval_results.diag_fwd_vel
diag_pelvis_h = eval_results.diag_pelvis_h
diag_l_foot = eval_results.diag_l_foot
diag_r_foot = eval_results.diag_r_foot
diag_energy = eval_results.diag_energy
diag_reward_components = eval_results.diag_reward_components
frames = getattr(eval_results, "frames", [])

print_eval_summary(eval_results, gate_passed, gate_failures, CURRENT_STAGE)

## Save Results & Stage Summary

Save structured training artifacts: stage summary text file, collected results
CSV (compatible with sweep analysis tooling), and model checkpoints.

In [ ]:
from environments.shared.reporting import save_jax_stage_artifacts

artifact_paths = save_jax_stage_artifacts(
    species=SPECIES,
    stage=CURRENT_STAGE,
    stage_config=_stage_cfg,
    stage_results=stage_results,
    stage_dir=STAGE_DIR,
    run_dir=RUN_DIR,
    eval_results=eval_results,
    final_eval_results=final_eval_results,
    params=jax.device_get(params),
    obs_rms=obs_rms,
    seed=TRAINING_SEED,
    num_envs=NUM_ENVS,
    reward_cfg=reward_cfg,
    best_params=best_params,
    best_reward=best_reward,
    best_update=best_update,
    evaluation_seed=EVALUATION_SEED,
)

# Print saved artifact paths
for name, path in artifact_paths.items():
    print(f"{name}: {path}")

# Print the stage summary inline
print()
print(artifact_paths["stage_summary"].read_text())

## 7. Training Curves & Locomotion Diagnostics

In [ ]:
# Training curves — uses shared plot_training_curves from jax_viz
curve_path = OUTPUT_DIR / "training_curves.png"

plot_training_curves(
    reward_history=reward_history,
    loss_history=loss_history,
    episode_return_history=episode_return_history,
    diagnostics_history=diagnostics_history,
    species=SPECIES,
    stage=CURRENT_STAGE,
    output_path=curve_path,
    show=True,
)

print(f"Training curves saved to: {curve_path}")

In [ ]:
# Locomotion diagnostics — uses shared plot_locomotion_diagnostics from jax_viz

plot_locomotion_diagnostics(
    eval_results,
    species=SPECIES,
    stage=CURRENT_STAGE,
    max_tilt_angle=MAX_TILT_ANGLE,
    healthy_z_range=(HEALTHY_Z_MIN, HEALTHY_Z_MAX),
    output_dir=STAGE_DIR,
    show=True,
)

print(f"Locomotion diagnostics saved to: {STAGE_DIR}")

### Reward Component Diagnostics

Per-component reward breakdown sampled during training.  
Shows which reward terms the agent is exploiting vs ignoring, plus body state variables vs termination thresholds.

In [ ]:
# Reward component breakdown over training
# Shows per-component reward contributions to diagnose exploit poses

if reward_component_history:
    diag_path = OUTPUT_DIR / "reward_components.png"
    plot_reward_components(
        reward_component_history,
        species=SPECIES,
        stage=CURRENT_STAGE,
        healthy_z_min=HEALTHY_Z_MIN,
        natural_forward_z=NATURAL_FORWARD_Z,
        nosedive_threshold=_NOSEDIVE_TERMINATION_THRESHOLD,
        output_path=diag_path,
        show=True,
    )
    print(f"Saved to {diag_path}")

    # Print final component breakdown
    print("\nFinal reward component means:")
    last = reward_component_history[-1]
    reward_keys = [k for k in last if not k.startswith("_") and k != "update"]
    total = 0.0
    for k in sorted(reward_keys):
        v = last.get(k, 0.0)
        total += v
        print(f"  {k:20s}: {v:+.4f}")
    print(f"  {'total':20s}: {total:+.4f}")
else:
    print("No reward component data collected. Re-run training with updated code.")

## 8. Record Training Video

Record a video of the best trained policy (highest reward during training) using
the CPU MuJoCo renderer. The JAX policy is evaluated deterministically (using the
action mean).

In [ ]:
try:
    import mediapy  # noqa: F401

    _HAS_MEDIAPY = True
except ImportError:
    _HAS_MEDIAPY = False
    print("mediapy not installed — installing now...")
    import subprocess

    subprocess.check_call(["pip", "install", "-q", "mediapy"])
    import mediapy  # noqa: F401

    _HAS_MEDIAPY = True
    print("mediapy installed successfully.")

if _HAS_MEDIAPY:
    video_params = best_params if best_params is not None else jax.device_get(params)
    print(f"Recording video with best model (update {best_update}, reward {best_reward:+.4f})")

    video_path = str(OUTPUT_DIR / "evaluation.mp4")

    frames, episode_reward = record_training_video(
        mj_model,
        video_params,
        network,
        obs_rms,
        get_obs_fn=get_obs,
        normalize_obs_fn=normalize_obs,
        scale_action_fn=scale_action,
        reward_fn=compute_reward,
        reward_cfg=reward_cfg,
        max_episode_steps=ctx.max_episode_steps,
        frame_skip=ctx.frame_skip,
        root_body_id=ctx.root_body_id,
        healthy_z_range=ctx.healthy_z_range,
        max_tilt_angle=ctx.max_tilt_angle,
        natural_forward_z=ctx.natural_forward_z,
        termination_body_heights=ctx.termination_body_heights,
        termination_site_heights=ctx.termination_site_heights,
        success_sites=ctx.success_sites,
        success_threshold=ctx.success_threshold,
        target_body="prey",
        sensor_quat_start=ctx.sensor_layout.quat_start,
        action_mapping=ctx.action_mapping,
        output_path=video_path,
        fps=50,
        camera_track_body=ctx.camera_track_body,
        camera_distance=ctx.camera_distance,
        show=True,
    )

    print(f"Episode reward: {episode_reward:.2f} | {len(frames)} frames")
    print(f"Saved to: {video_path}")

In [ ]:
# Create a frame collage for quick visual review
# Single image with labelled frame numbers and timestamps

collage_path = STAGE_DIR / "eval_collage.png"
collage_fig = create_frame_collage(
    frames,
    output_path=collage_path,
    num_frames=10,
    cols=5,
    title=f"{SPECIES.capitalize()} Stage {CURRENT_STAGE} — Eval Rollout ({len(frames)} frames, {len(frames) / 50:.1f}s)",
    fps=50,
    show=True,
)
print(f"Collage saved to: {collage_path}")

# Auto-download in Google Colab
try:
    from google.colab import files

    files.download(str(collage_path))
except ImportError:
    pass

In [ ]:
# Optional plots/video/collage add files after the early bundle export.
# Refresh the manifest last while preserving its partial/failed/complete status.
import json

from environments.shared.result_bundle import validate_result_bundle, write_artifact_manifest

_manifest_path = RUN_DIR / "artifact_manifest.json"
if not _manifest_path.is_file():
    raise FileNotFoundError(f"Early artifact manifest is missing: {_manifest_path}")
_manifest_status = json.loads(_manifest_path.read_text()).get("status")
if _manifest_status not in {"partial", "failed", "complete"}:
    raise ValueError(f"Cannot preserve invalid manifest status: {_manifest_status!r}")

FINAL_MANIFEST_PATH = write_artifact_manifest(RUN_DIR, status=_manifest_status)
FINAL_BUNDLE_REPORT = validate_result_bundle(
    RUN_DIR,
    require_complete=_manifest_status == "complete",
)
print(f"Final manifest refreshed ({_manifest_status}): {FINAL_MANIFEST_PATH}")
print(f"Bundle validation: {FINAL_BUNDLE_REPORT['status']}")

## 9. Next Steps

To continue curriculum training, keep the same `RUN_ID`, change `CURRENT_STAGE`,
and re-run from the configuration cell. The reward config is loaded automatically
from `configs/<species>/`:

```python
CURRENT_STAGE = 2  # or 3
```

For Stage 2 or 3, `RESUME_FROM = None` automatically loads the previous stage's
`models/best_model.pkl` only when its `stage_result.json` exists and records
`publication_gate_passed: true`. Missing or failed prior stages stop before training.

**Explicit checkpoint override:** Relative paths are resolved from `RUN_DIR`,
not from the new stage directory. For example:

```python
RESUME_FROM = "stage1/models/best_model.pkl"
```

A fresh Colab session must use the same Git commit and dependency versions
captured in `provenance.json`; incompatible code or runtime state is rejected
before the stages can be combined.

To train a different species, change `SPECIES` in the configuration cell and
restart from the beginning.

## 10. Auto-Disconnect (Optional)

Optionally disconnect the Colab runtime after completion to free up resources.
Set `AUTO_DISCONNECT = True` in the configuration cell or toggle below.

In [ ]:
AUTO_DISCONNECT = True  # Set to True to disconnect runtime after training

if AUTO_DISCONNECT:
    import time

    if USE_GOOGLE_DRIVE:
        from google.colab import drive

        print("Flushing Google Drive writes before disconnect...")
        drive.flush_and_unmount()
    print("Training finished. Disconnecting runtime in 5 seconds...")
    time.sleep(5)
    from google.colab import runtime

    runtime.unassign()
else:
    print("Training finished. Runtime kept alive — remember to disconnect manually when done.")